# Tome C3, Chapter 6: Completeness and Compactness (Python companion)

Exercises first, solutions at the end. We write first-order formulas as Python data, evaluate them in finite structures, and search for small models and countermodels. We build a tableau prover that finds proofs of valid sentences and runs forever on the others, as Church's theorem says it must. Then we run Cantor's back-and-forth construction, look at a structure that imitates the integers and fails, and test the two halves of the Ax–Grothendieck proof, finite fields and the Lefschetz principle, on actual polynomials.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random, time
from fractions import Fraction as Fr
import matplotlib.pyplot as plt
random.seed(6)

## Exercises

**P6.1 (★) Formulas and finite structures.** Represent terms and formulas as nested tuples, and write `holds(M, phi, s)` deciding $\mathcal M\models\varphi[s]$ for a finite structure $\mathcal M$. Check that the drinker's sentence of Exercise 6.1 holds in every structure with at most $3$ elements, that $\forall x\,\exists y\,(x\cdot y=e)$ holds in $\mathbb Z/n\mathbb Z$, and that the density sentence of Tome C1, Chapter 1 fails in every finite linear order with two or more elements.

**P6.2 (★) Searching for small models.** For each of the following sets of sentences, find the smallest model, or check that there is none up to a bound: a function that is injective but not surjective; a strict linear order with no largest element; a triangle-free graph in which every vertex has exactly three neighbours; and a countermodel to $\forall y\,\exists x\,R(x,y)\to\exists x\,\forall y\,R(x,y)$. Which ones have only infinite models?

**P6.3 (★★) A tableau prover.** Write a prover for first-order logic without equality: signed tableaux with the propositional rules, new constants for $\exists$ (the Henkin witnesses of Theorem 6.8), and instantiation of $\forall$ by the terms on the branch, under a budget that grows. Prove the drinker's principle, axiom (A3), $\exists x\,\forall y\,R(x,y)\to\forall y\,\exists x\,R(x,y)$, $\forall x\,(P(x)\to P(f(x)))\to(P(c)\to P(f(f(f(c)))))$, and the barber's sentence $\neg\exists x\,\forall y\,(R(y,x)\leftrightarrow\neg R(y,y))$.

**P6.4 (★★) Semi-decision.** Run the prover of P6.3 on the converse $\forall y\,\exists x\,R(x,y)\to\exists x\,\forall y\,R(x,y)$ with increasing budgets, and measure how the work grows. Explain, with Theorem 6.10, why no budget can be chosen in advance for all sentences, and what P6.2 adds.

**P6.5 (★) Back and forth.** Enumerate the positive rationals with the Calkin–Wilf sequence, map them to all of $\mathbb Q$, enumerate the dyadic rationals of $]0,1[$, and run the proof of Theorem 6.17 for $40$ steps. Check that the result is order-preserving, and reproduce the pairs of Figure 6.1 with its enumerations.

**P6.6 (★★) An impostor of $\mathbb N$.** The polynomials $a_dt^d+\dots+a_0\in\mathbb Z[t]$ with positive leading coefficient, together with $0$, ordered by the sign of the difference's leading coefficient, form a discretely ordered semiring with infinite elements such as $t$. Check some sentences of $\mathrm{Th}(\mathbb N)$ in it on samples, find the galaxies of $t$, $2t$, $t^2$, and show that it is not a model of $\mathrm{Th}(\mathbb N)$ by a sentence used in the proof of Theorem 6.18.

**P6.7 (★★) The finite-field half of Ax–Grothendieck.** Sample polynomial maps $\mathbb F_p^2\to\mathbb F_p^2$ and check that the injective ones are surjective. Then explain why the proof of Theorem 6.28 needs injectivity on all of $K_p$, not just on $\mathbb F_p^n$: find $p$ and a polynomial that permutes $\mathbb F_p$ but is not injective on $\mathbb F_{p^2}$. Find also an injective polynomial map $\mathbb Q\to\mathbb Q$ that is not surjective.

**P6.8 (★★) The Lefschetz principle at work.** The sentence ``$x^2+x+1=0$ has two distinct roots'' is true in $\mathbb C$. Implement $\mathbb F_{p^2}$, in which every quadratic over $\mathbb F_p$ has its roots, and find the primes for which the sentence fails in characteristic $p$. Do the same for ``$x^3=2$ has three distinct roots'' (use a field where these roots live), and relate the exceptions to Theorem 6.27(c).

## Solutions

### P6.1

Terms are variables (strings) or tuples `('f', t1, ..., tk)`; constants are `('c',)`. Formulas are tuples with a head: `'R'` for a relation, `'='`, `'not'`, `'and'`, `'or'`, `'imp'`, `'all'`, `'ex'`. A structure is a domain with dictionaries of functions and relations.

In [ ]:
def Rel(name, *ts): return ('R', name) + ts
def Eq(a, b): return ('=', a, b)
def Not(a): return ('not', a)
def And(*a): return a[0] if len(a) == 1 else ('and', a[0], And(*a[1:]))
def Or(*a): return a[0] if len(a) == 1 else ('or', a[0], Or(*a[1:]))
def Imp(a, b): return ('imp', a, b)
def All(x, a): return ('all', x, a)
def Ex(x, a): return ('ex', x, a)
def Fn(name, *ts): return (name,) + ts

class Structure:
    def __init__(self, n, funs=None, rels=None):
        self.dom = range(n)
        self.funs = funs or {}
        self.rels = rels or {}

def val(M, t, s):
    if isinstance(t, str):
        return s[t]
    return M.funs[t[0]](*[val(M, u, s) for u in t[1:]])

def holds(M, f, s=None):
    s = s or {}
    h = f[0]
    if h == 'R':
        return tuple(val(M, t, s) for t in f[2:]) in M.rels[f[1]]
    if h == '=':
        return val(M, f[1], s) == val(M, f[2], s)
    if h == 'not':
        return not holds(M, f[1], s)
    if h == 'and':
        return holds(M, f[1], s) and holds(M, f[2], s)
    if h == 'or':
        return holds(M, f[1], s) or holds(M, f[2], s)
    if h == 'imp':
        return (not holds(M, f[1], s)) or holds(M, f[2], s)
    if h == 'all':
        return all(holds(M, f[2], {**s, f[1]: m}) for m in M.dom)
    if h == 'ex':
        return any(holds(M, f[2], {**s, f[1]: m}) for m in M.dom)
    raise ValueError(h)

drinker = Ex('x', Imp(Rel('D', 'x'), All('y', Rel('D', 'y'))))
count = 0
for n in range(1, 4):
    for k in range(n + 1):
        for D in itertools.combinations(range(n), k):
            M = Structure(n, rels={'D': {(d,) for d in D}})
            assert holds(M, drinker)
            count += 1
print("drinker's sentence holds in all", count, "structures with at most 3 elements")

inverse = All('x', Ex('y', Eq(Fn('*', 'x', 'y'), Fn('e'))))
for n in range(1, 8):
    Zn = Structure(n, funs={'*': lambda a, b, n=n: (a + b) % n, 'e': lambda: 0})
    assert holds(Zn, inverse)
print("every element of Z/nZ has an inverse, n = 1..7")

dense = All('x', All('y', Imp(Rel('<', 'x', 'y'), Ex('z', And(Rel('<', 'x', 'z'), Rel('<', 'z', 'y'))))))
for n in range(1, 8):
    L = Structure(n, rels={'<': {(a, b) for a in range(n) for b in range(n) if a < b}})
    print(n, "elements: dense?", holds(L, dense))

A one-element order is (vacuously) dense; from two elements on, the two neighbours $0<1$ have nothing between them. A dense order with two elements must be infinite, which a finite search can only suggest and compactness explains.

### P6.2

We enumerate all structures of each size: all functions $n\to n$, all relations, or all graphs.

In [ ]:
def search(theory, structures, nmax):
    for n in range(1, nmax + 1):
        tried = 0
        for M in structures(n):
            tried += 1
            if all(holds(M, phi) for phi in theory):
                return n, M, tried
    return None

def all_functions(n):
    for images in itertools.product(range(n), repeat=n):
        yield Structure(n, funs={'f': lambda a, im=images: im[a]})

def all_binary(n, name):
    pairs = [(a, b) for a in range(n) for b in range(n)]
    for bits in itertools.product([0, 1], repeat=len(pairs)):
        yield Structure(n, rels={name: {p for p, b in zip(pairs, bits) if b}})

def all_graphs(n):
    pairs = list(itertools.combinations(range(n), 2))
    for bits in itertools.product([0, 1], repeat=len(pairs)):
        E = set()
        for (a, b), x in zip(pairs, bits):
            if x:
                E |= {(a, b), (b, a)}
        yield Structure(n, rels={'E': E})

f = lambda t: Fn('f', t)
inj_not_surj = [All('x', All('y', Imp(Eq(f('x'), f('y')), Eq('x', 'y')))),
                Ex('y', All('x', Not(Eq(f('x'), 'y'))))]
lt = lambda a, b: Rel('<', a, b)
order_no_max = [All('x', Not(lt('x', 'x'))),
                All('x', All('y', All('z', Imp(And(lt('x', 'y'), lt('y', 'z')), lt('x', 'z'))))),
                All('x', Ex('y', lt('x', 'y')))]
E = lambda a, b: Rel('E', a, b)
nbrs3 = Ex('a', Ex('b', Ex('c', And(Not(Eq('a', 'b')), Not(Eq('a', 'c')), Not(Eq('b', 'c')),
           E('x', 'a'), E('x', 'b'), E('x', 'c'),
           All('d', Imp(E('x', 'd'), Or(Eq('d', 'a'), Eq('d', 'b'), Eq('d', 'c'))))))))
cubic_trianglefree = [All('x', nbrs3),
                      Not(Ex('x', Ex('y', Ex('z', And(E('x', 'y'), E('y', 'z'), E('z', 'x'))))))]
R = lambda a, b: Rel('R', a, b)
counter = [All('y', Ex('x', R('x', 'y'))), Not(Ex('x', All('y', R('x', 'y'))))]

t0 = time.time()
for name, th, gen, nmax in [("injective, not surjective f", inj_not_surj, all_functions, 5),
                            ("strict order, no largest element", order_no_max, lambda n: all_binary(n, '<'), 3),
                            ("cubic triangle-free graph", cubic_trianglefree, all_graphs, 6),
                            ("countermodel to ∀y∃xR → ∃x∀yR", counter, lambda n: all_binary(n, 'R'), 3)]:
    r = search(th, gen, nmax)
    if r is None:
        print(f"{name:34s}: no model with at most {nmax} elements")
    else:
        n, M, tried = r
        print(f"{name:34s}: smallest model has {n} elements (found after {tried} tries); relations {M.rels}")
print(f"{time.time() - t0:.1f} s")

The first two theories have no finite model at all, by a one-line argument each (an injective map of a finite set to itself is onto, the pigeonhole principle; a finite non-empty order has a largest element); they have infinite models, $x\mapsto x+1$ and $(\mathbb N,<)$. The cubic triangle-free graph appears at $6$ vertices: it is $K_{3,3}$. The countermodel is the $2$-cycle $R=\{(0,1),(1,0)\}$: everyone has someone pointing at them, but no one points at everyone. A finite countermodel is a certificate of non-validity; P6.4 shows why one cannot always hope for it.

### P6.3

Signed formulas $(\text{sign},\varphi)$: $(T,\varphi)$ asserts $\varphi$, $(F,\varphi)$ denies it. A branch closes when it contains $(T,\alpha)$ and $(F,\alpha)$ for an atom $\alpha$. The rule for $(T,\exists x\varphi)$ and $(F,\forall x\varphi)$ adds $\varphi[x:=c]$ for a new constant $c$, a Henkin witness; the rule for $(T,\forall x\varphi)$ and $(F,\exists x\varphi)$ may be used again and again, with every term on the branch, and a budget bounds these uses.

In [ ]:
def subst(f, x, t):
    # substitute the closed term t for the free variable x
    h = f[0]
    if h == 'R':
        return f[:2] + tuple(subst_t(u, x, t) for u in f[2:])
    if h in ('all', 'ex'):
        return f if f[1] == x else (h, f[1], subst(f[2], x, t))
    return (h,) + tuple(subst(g, x, t) for g in f[1:])

def subst_t(u, x, t):
    if isinstance(u, str):
        return t if u == x else u
    return (u[0],) + tuple(subst_t(v, x, t) for v in u[1:])

def terms_of(f, acc):
    if f[0] == 'R':
        for u in f[2:]:
            collect_terms(u, acc)
    elif f[0] in ('all', 'ex'):
        terms_of(f[2], acc)
    else:
        for g in f[1:]:
            terms_of(g, acc)

def collect_terms(u, acc):
    if not isinstance(u, str):
        acc.add(u)
        for v in u[1:]:
            collect_terms(v, acc)

stats = {'nodes': 0, 'inst': 0}

def tableau(todo, atoms, gammas, terms, budget, fresh):
    # todo: list of signed formulas to expand; atoms: set of signed atoms; gammas: list of (sign, f, used terms)
    stats['nodes'] += 1
    todo = list(todo)
    while todo:
        sg, f = todo.pop()
        h = f[0]
        if h == 'R':
            if (not sg, f) in atoms:
                return True
            atoms = atoms | {(sg, f)}
            acc = set(); terms_of(f, acc); terms = terms | acc
        elif h == 'not':
            todo.append((not sg, f[1]))
        elif (h, sg) in (('and', True), ('or', False)):
            todo += [(sg, f[1]), (sg, f[2])]
        elif (h, sg) == ('imp', False):
            todo += [(True, f[1]), (False, f[2])]
        elif (h, sg) in (('and', False), ('or', True), ('imp', True)):
            left = (sg, f[1]) if h != 'imp' else (False, f[1])
            right = (sg, f[2])
            return (tableau(todo + [left], atoms, gammas, terms, budget, fresh) and
                    tableau(todo + [right], atoms, gammas, terms, budget, fresh))
        elif (h, sg) in (('ex', True), ('all', False)):
            c = ('c%d' % fresh[0],); fresh = (fresh[0] + 1,)
            terms = terms | {c}
            todo.append((sg, subst(f[2], f[1], c)))
        else:   # ('all', True) or ('ex', False): remember it
            gammas = gammas + [(sg, f, frozenset())]
    # all non-gamma formulas are expanded: instantiate the gamma formulas
    if budget == 0 or not gammas:
        return False
    if not terms:
        terms = frozenset({('c%d' % fresh[0],)}); fresh = (fresh[0] + 1,)
    new_todo, new_gammas = [], []
    for sg, f, used in gammas:
        fresh_terms = [t for t in sorted(terms, key=str) if t not in used]
        new_todo += [(sg, subst(f[2], f[1], t)) for t in fresh_terms]
        new_gammas.append((sg, f, used | frozenset(fresh_terms)))
    if not new_todo:
        return False
    stats['inst'] += len(new_todo)
    return tableau(new_todo, atoms, new_gammas, terms, budget - 1, fresh)

def prove(f, budget=4):
    stats['nodes'] = stats['inst'] = 0
    return tableau([(False, f)], frozenset(), [], frozenset(), budget, (0,))

P = lambda a: Rel('P', a); Q = lambda a: Rel('Q', a); D = lambda a: Rel('D', a)
tests = {
    "drinker ∃x(D(x)→∀yD(y))": drinker,
    "(A3) ∀x(P→Q)→(∀xP→∀xQ)": Imp(All('x', Imp(P('x'), Q('x'))), Imp(All('x', P('x')), All('x', Q('x')))),
    "∃x∀yR(x,y)→∀y∃xR(x,y)": Imp(Ex('x', All('y', R('x', 'y'))), All('y', Ex('x', R('x', 'y')))),
    "∀x(P(x)→P(f(x)))→(P(c)→P(f(f(f(c)))))": Imp(All('x', Imp(P('x'), P(f('x')))),
                                              Imp(P(('c',)), P(f(f(f(('c',))))))),
    "¬∃x∀y(R(y,x)↔¬R(y,y)) (barber)": Not(Ex('x', All('y', And(Imp(R('y', 'x'), Not(R('y', 'y'))),
                                                                Imp(Not(R('y', 'y')), R('y', 'x')))))),
}
for name, phi in tests.items():
    t0 = time.time()
    r = prove(phi)
    print(f"{name:42s} proved: {r}   ({stats['nodes']} nodes, {time.time() - t0:.3f} s)")

All five are proved. The barber sentence (no one shaves exactly those who do not shave themselves) is Russell's paradox in first-order form: instantiating $\forall y$ with the barber $x$ himself closes every branch. The prover is a direct transcription of the completeness proof: an open branch on which every rule has been applied fairly describes a Henkin model, built from the terms on the branch, so a fair search that never closes all branches has found (in the limit) a model of the negation. Completeness says that for a valid sentence the tableau closes after finitely many steps.

### P6.4

In [ ]:
converse = Imp(All('y', Ex('x', R('x', 'y'))), Ex('x', All('y', R('x', 'y'))))
rows = []
for b in range(1, 9):
    t0 = time.time()
    r = prove(converse, budget=b)
    rows.append((b, stats['inst'], time.time() - t0))
    print(f"budget {b}: proved {r}, {stats['inst']} instances of ∀ tried, {time.time() - t0:.3f} s")
plt.figure(figsize=(5, 3))
plt.semilogy([r[0] for r in rows], [r[1] for r in rows], 'o-')
plt.xlabel("budget of ∀-instantiation rounds"); plt.ylabel("instances tried")
plt.title("searching for a proof that does not exist"); plt.tight_layout(); plt.show()

Each round introduces new witnesses, which create new terms, which feed new instances: the branch never closes, and the number of instances tried grows with the budget without end: here it doubles with each round, as every round creates as many new terms as it tries instances. For this sentence, P6.2 found a finite countermodel, which settles it. But Theorem 6.10 says that no algorithm settles every sentence: if, for each sentence, either a proof or a finite countermodel existed, running both searches in parallel would decide validity. So there are non-valid sentences whose countermodels are all infinite (P6.2 found two theories with only infinite models), and on their negations the search simply runs. A budget is a guess, and no computable guess is always right.

### P6.5

The Calkin–Wilf sequence $q_1=1$, $q_{n+1}=1/(2\lfloor q_n\rfloor-q_n+1)$ lists every positive rational exactly once; $0,q_1,-q_1,q_2,-q_2,\dots$ then lists $\mathbb Q$.

In [ ]:
def rationals():
    yield Fr(0)
    q = Fr(1)
    while True:
        yield q; yield -q
        q = 1 / (2 * math.floor(q) - q + 1)

def dyadics():
    k = 1
    while True:
        for j in range(1, 2 ** k, 2):
            yield Fr(j, 2 ** k)
        k += 1

def back_and_forth(A, B, steps):
    A_list, B_list = [], []
    Ait, Bit = iter(A), iter(B)
    def ensure(lst, it, n):
        while len(lst) < n:
            lst.append(next(it))
    pairs = []
    def first_in_gap(p, x, lst, it):
        lo = [b for a, b in p if a < x]; hi = [b for a, b in p if a > x]
        used = {b for _, b in p}
        i = 0
        while True:
            ensure(lst, it, i + 1)
            t = lst[i]
            if t not in used and (not lo or t > max(lo)) and (not hi or t < min(hi)):
                return t
            i += 1
    for k in range(steps):
        if k % 2 == 0:
            i = 0
            while True:
                ensure(A_list, Ait, i + 1)
                if A_list[i] not in {a for a, _ in pairs}: break
                i += 1
            a = A_list[i]; b = first_in_gap(pairs, a, B_list, Bit)
        else:
            i = 0
            while True:
                ensure(B_list, Bit, i + 1)
                if B_list[i] not in {b for _, b in pairs}: break
                i += 1
            b = B_list[i]; a = first_in_gap([(y, x) for x, y in pairs], b, A_list, Ait)
        pairs.append((a, b))
    return pairs

pairs = back_and_forth(rationals(), dyadics(), 40)
print("first pairs:", [(str(a), str(b)) for a, b in pairs[:10]])
ok = all((a1 < a2) == (b1 < b2) for (a1, b1), (a2, b2) in itertools.combinations(pairs, 2))
print("order-preserving on 40 pairs:", ok)
fig_A = [Fr(0), Fr(1), Fr(-1), Fr(1, 2), Fr(2), Fr(-2), Fr(-1, 2), Fr(3, 2)]
fig_B = [Fr(1, 2), Fr(1, 4), Fr(3, 4), Fr(1, 8), Fr(3, 8), Fr(5, 8), Fr(7, 8), Fr(1, 16)]
print("Figure 6.1:", [(str(a), str(b)) for a, b in back_and_forth(fig_A, fig_B, 7)])
xs, ys = zip(*sorted(pairs))
plt.figure(figsize=(5, 3.2))
plt.plot([float(x) for x in xs], [float(y) for y in ys], 'o-', ms=3)
plt.xlabel("rational"); plt.ylabel("dyadic in ]0,1["); plt.title("40 steps of an isomorphism Q → D")
plt.tight_layout(); plt.show()

The graph is increasing, as it must be, and it keeps filling in: after $n$ steps, the first $n/2$ elements of each enumeration are matched. The limit is an order isomorphism $\mathbb Q\to D$, which no formula gives: it depends on the enumerations. With the enumerations of the figure, the program reproduces its seven pairs.

### P6.6

An element is a tuple of coefficients $(a_0,\dots,a_d)$; $x<y$ when the leading coefficient of $y-x$ is positive.

In [ ]:
def norm(p):
    p = list(p)
    while p and p[-1] == 0:
        p.pop()
    return tuple(p)
def add(p, q):
    n = max(len(p), len(q))
    return norm([(p[i] if i < len(p) else 0) + (q[i] if i < len(q) else 0) for i in range(n)])
def neg(p): return tuple(-a for a in p)
def mul(p, q):
    if not p or not q: return ()
    r = [0] * (len(p) + len(q) - 1)
    for i, a in enumerate(p):
        for j, b in enumerate(q):
            r[i + j] += a * b
    return norm(r)
def less(p, q):
    d = add(q, neg(p))
    return bool(d) and d[-1] > 0
def nonneg(p): return not p or p[-1] > 0
def show(p):
    if not p: return "0"
    out = ""
    for i in reversed(range(len(p))):
        a = p[i]
        if a == 0: continue
        mon = "" if i == 0 else "t" if i == 1 else f"t^{i}"
        coef = str(abs(a)) if (abs(a) != 1 or i == 0) else ""
        sign = ("-" if a < 0 else "") if not out else (" - " if a < 0 else " + ")
        out += sign + coef + mon
    return out

one, t = (1,), (0, 1)
sample = [norm(c) for c in itertools.product(range(-2, 3), repeat=3) if nonneg(norm(c))]
print(len(sample), "sample elements, e.g.", [show(p) for p in sample[:6]])
# discreteness: no element strictly between x and x+1
disc = all(not (less(x, y) and less(y, add(x, one))) for x in sample for y in sample)
# x < y implies x + z = y for some z >= 0 (namely z = y - x, which is in the semiring)
diff = all(nonneg(add(y, neg(x))) for x in sample for y in sample if less(x, y))
print("discrete order on the sample:", disc, "  differences exist:", diff)
# galaxies: a and b are in the same galaxy iff b - a is a constant
def same_galaxy(a, b):
    d = add(b, neg(a)); return len(d) <= 1
for a, b in [(t, add(t, (5,))), (t, mul((2,), t)), (mul((2,), t), mul(t, t)), (add(t, (-7,)), t)]:
    print(f"{show(a):>8s} and {show(b):<8s} same galaxy: {same_galaxy(a, b)}")
# the sentence  ∀x ∃y (x = y + y  or  x = y + y + 1)  fails at x = t
def halves(x, bound=3):
    for c in itertools.product(range(-bound, bound + 1), repeat=max(len(x), 1)):
        y = norm(c)
        if nonneg(y) and (add(y, y) == x or add(add(y, y), one) == x):
            return y
    return None
print("t is even or odd?", halves(t), "  2t + 1 is odd with half", show(halves(add(mul((2,), t), one))))

The structure agrees with $\mathbb N$ on the sentences tested, and it has a standard part followed by galaxies $t+\mathbb Z$, $2t+\mathbb Z$, $t^2+\mathbb Z$. But $t$ is neither $y+y$ nor $y+y+1$ for any $y$ (the leading coefficient of $y+y$ is even, and the search confirms there is no small $y$), so the sentence $\forall x\,\exists y\,(x=y+y\lor x=y+y+1)$, true in $\mathbb N$, fails. This is exactly the sentence that gave density and the absence of a smallest galaxy in Theorem 6.18: here the galaxy of $t$ is the smallest non-standard galaxy (an element above every integer and below $t-n$ for every $n$ would have degree $1$ and leading coefficient strictly between $0$ and $1$), which Theorem 6.18 forbids, and it is halving that fails. Real non-standard models cannot be written down this simply, and Tennenbaum's theorem says they cannot be written down by any program.

### P6.7

In [ ]:
def poly_map_eval(F, x, y, p):
    # F: list of two dicts {(i, j): coeff}
    return tuple(sum(c * pow(x, i, p) * pow(y, j, p) for (i, j), c in Fi.items()) % p for Fi in F)

def random_map(p, d):
    mons = [(i, j) for i in range(d + 1) for j in range(d + 1 - i)]
    return [{m: random.randrange(p) for m in mons if random.random() < 0.5} for _ in range(2)]

for p, d in [(3, 2), (5, 2), (7, 3)]:
    inj = 0
    for _ in range(3000):
        F = random_map(p, d)
        image = {poly_map_eval(F, x, y, p) for x in range(p) for y in range(p)}
        if len(image) == p * p:
            inj += 1
    print(f"p={p}, degree ≤ {d}: {inj} of 3000 sampled maps are injective, and all of these are onto (image size {p*p})")

# x^3 permutes F_p iff gcd(3, p-1) = 1; on F_{p^2} it is never injective since 3 | p^2 - 1
for p in [5, 11, 17]:
    perm = len({pow(x, 3, p) for x in range(p)}) == p
    print(f"x^3 permutes F_{p}: {perm};  gcd(3, {p*p}-1) = {math.gcd(3, p * p - 1)}, so x^3 is 3-to-1 on the units of F_{p*p}")
cubes = {Fr(a, b) ** 3 for a in range(-6, 7) for b in range(1, 7)}
print("x -> x^3 on Q: injective (it is increasing), and 2 is a cube?", Fr(2) in cubes, "(and never is: 2^(1/3) is irrational)")

Over a finite set, injective implies onto by counting, so the first check can never fail; that is the whole point. The cube map shows the subtlety. For $p=5$, $x\mapsto x^3$ is a bijection of $\mathbb F_5$, since $\gcd(3,4)=1$; but $\mathbb F_{25}^*$ is cyclic of order $24$, divisible by $3$, so $x^3$ is $3$-to-$1$ on it. The map is not injective on $K_5$, and the sentence $\sigma_{1,3}$ is not contradicted. This is why the proof of Theorem 6.28 assumes injectivity on $K_p^n$ and then restricts to a finite subfield $E$ that contains the coefficients and the target: injectivity on $K_p^n$ gives it on $E^n$. Over $\mathbb Q$, which is not algebraically closed, the cube map is injective and misses $2$.

### P6.8

$\mathbb F_{p^2}=\mathbb F_p[x]/(f)$ for an irreducible quadratic $f$; every quadratic over $\mathbb F_p$ splits there. For cube roots of $2$ we use $\mathbb F_{p^2}$ or $\mathbb F_{p^3}$, built from an irreducible polynomial of degree $2$ or $3$ (which is irreducible as soon as it has no root in $\mathbb F_p$).

In [ ]:
def trim(a):
    a = list(a)
    while a and a[-1] == 0:
        a.pop()
    return a
def pmod(a, f, p):
    # remainder of a modulo the monic polynomial f, coefficients low to high
    a = [x % p for x in a]
    while len(trim(a)) >= len(f):
        a = trim(a); c = a[-1]; s = len(a) - len(f)
        for i, fi in enumerate(f):
            a[s + i] = (a[s + i] - c * fi) % p
    return trim(a)
def pmul(a, b, f, p):
    r = [0] * (len(a) + len(b))
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            r[i + j] += x * y
    return pmod(r, f, p)
def ppow(a, e, f, p):
    r = [1]
    while e:
        if e & 1: r = pmul(r, a, f, p)
        a = pmul(a, a, f, p); e >>= 1
    return r
def has_root_in_Fp(f, p):
    return any(sum(c * pow(x, i, p) for i, c in enumerate(f)) % p == 0 for x in range(p))
def irreducible(p, k):
    # degree 2 or 3: irreducible iff no root in F_p
    assert k in (2, 3)
    for c in itertools.product(range(p), repeat=k):
        f = list(c) + [1]
        if not has_root_in_Fp(f, p):
            return f
def distinct_roots(poly, p, k):
    # roots of an integer polynomial (low to high) in F_{p^k}, k in {1, 2, 3}
    f = [0, 1] if k == 1 else irreducible(p, k)
    roots = 0
    for c in itertools.product(range(p), repeat=k):
        a = trim(c)
        v = []
        for coef in reversed(poly):          # Horner: v = v*a + coef
            v = pmul(v, a, f, p)
            v = pmod([(v[0] if v else 0) + coef] + v[1:], f, p)
        roots += (v == [])
    return roots

primes = [q for q in range(2, 32) if all(q % r for r in range(2, q))]
bad1 = [p for p in primes if distinct_roots([1, 1, 1], p, 2) != 2]
def k_for(p):
    return 1 if p == 3 else 2 if p % 3 == 2 else 3
bad2 = [p for p in primes if distinct_roots([-2, 0, 0, 1], p, k_for(p)) != 3]
print("x^2+x+1 has two distinct roots in characteristic p, except p =", bad1)
print("x^3-2  has three distinct roots in characteristic p, except p =", bad2)

The exceptions are where the discriminants vanish: $x^2+x+1$ has discriminant $-3$, and in characteristic $3$ it is $(x-1)^2$; $x^3-2$ has discriminant $-108=-2^2\cdot3^3$, and in characteristics $2$ and $3$ it becomes $x^3$ and $(x+1)^3$, with a single root. (We searched in a finite field containing all three roots when they are distinct: $\mathbb F_{p^2}$ when $p\equiv2\pmod 3$, where $2$ has a cube root in $\mathbb F_p$ and the cube roots of unity lie in $\mathbb F_{p^2}$; $\mathbb F_{p^3}$ when $p\equiv1\pmod3$, where the cube roots of unity lie in $\mathbb F_p$ and $x^3-2$ splits in $\mathbb F_{p^3}$, being either split already or irreducible.) Theorem 6.27 predicted the shape of the answer: a sentence true in $\mathbb C$ is true in every algebraically closed field of characteristic $p>N$, with finitely many exceptions, here the primes dividing a discriminant, that a proof of the sentence from $\mathrm{ACF}_0$ would have to mention through the axioms $\overline p\ne0$.